<h1>このプロジェクトでのデータ処理によるデータの変化をたどっていく。</h1>
<h3>ここではDiverseVulから提供されているデータセットを使う。train/val/testはすでに分割された状態で提供されており、testにはtrain, valに含まれていないプロジェクトからのデータを使用することで本番環境のようなunseenデータでの性能を図ることを目的としている</h3>

<h3>以下の順番で確認していく(run.pyに準拠)</h3>
<h3>
<ol>
<li>生のjsonファイルの中身確認</li>
<li>run.pyでのselect関数とdata.clean後のDataFrameの確認</li>
<li>CPG pklの中身の確認</li>
<li>tokens pklの中身の確認</li>
<li>input pkl (PyG Data)の確認</li>
<li>StreamGraphDataset/BalancedStreamDatasetから出てくるバッチの確認</li>
</ol>

</h3>

<h2>環境の準備</h2>

In [24]:
import os
import sys

project_root = "/home/yudai/Project/research/Vul_Detection"

# CWD を合わせる（念のため）
os.chdir(project_root)

# すでに sys.path に入っていたら一度消してから先頭に入れる
if project_root in sys.path:
    sys.path.remove(project_root)
sys.path.insert(0, project_root)

print("CWD:", os.getcwd())
print("sys.path[0]:", sys.path[0])


CWD: /home/yudai/Project/research/Vul_Detection
sys.path[0]: /home/yudai/Project/research/Vul_Detection


In [25]:

import os
from pathlib import Path

import pandas as pd
import torch

import configs
import utils.data as data
import utils.process as process
from utils.functions.cpg import parse_to_functions
from utils.process.embeddings import nodes_to_input
from utils.validate.analyze_utils import describe_pyg_graph

# run.py から再利用したい関数・クラスをインポート
from run import (
    train_path, valid_path, test_path,
    select,
    StreamGraphDataset, BalancedStreamDataset,
    pyg_collate,
    _split_file_paths,  # 先頭_ですが import はできます
)


PATHS = configs.Paths()
FILES = configs.Files()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

<h2>１：生のJsonファイルの中身確認</h2>

In [26]:

def show_basic_df_stats(df: pd.DataFrame, name: str, n_head: int = 5):
    print(f"===== {name} =====")
    print("shape:", df.shape)
    print("columns:", list(df.columns))
    print("\n--- head() ---")
    display(df.head(n_head))
    if "target" in df.columns:
        print("\n--- target value_counts ---")
        print(df["target"].value_counts(dropna=False))
    if "func" in df.columns:
        print("\n--- func length describe ---")
        print(df["func"].str.len().describe())
    print("\n============================\n")


In [27]:
# セル3: 生のJSONLを確認（train/valid/test）
train_raw = pd.read_json(train_path, lines=True)
valid_raw = pd.read_json(valid_path, lines=True)
test_raw  = pd.read_json(test_path,  lines=True)

show_basic_df_stats(train_raw, "train_raw")
show_basic_df_stats(valid_raw, "valid_raw")
show_basic_df_stats(test_raw,  "test_raw")


===== train_raw =====
shape: (434696, 5)
columns: ['project', 'commit_id', 'target', 'func', 'idx']

--- head() ---


,project,commit_id,target,func,idx
0,debian,None,1,"extern int name ( int , locale_t ) __THROW __e...",0
1,debian,None,1,unsigned long # define BN_LONG long # define B...,1
2,debian,None,1,WRITE_CLASS_ENCODER ( CephXServiceTicket ) str...,2
3,chrome,None,1,void vp9_init_layer_context ( VP9_COMP * const...,3
4,debian,None,1,static void rv40_loop_filter ( RV34DecContext ...,4



--- target value_counts ---
target
0    399056
1     35640
Name: count, dtype: int64

--- func length describe ---
count    434696.000000
mean       1252.254978
std        3759.277196
min           0.000000
25%         223.000000
50%         489.000000
75%        1158.000000
max      484356.000000
Name: func, dtype: float64


===== valid_raw =====
shape: (48539, 5)
columns: ['project', 'commit_id', 'target', 'func', 'idx']

--- head() ---


,project,commit_id,target,func,idx
0,chrome,None,1,static void set_fixed_partitioning ( VP9_COMP ...,23
1,chrome,None,1,"static void _HZOpen ( UConverter * cnv , UConv...",25
2,debian,None,1,enum message_cte message_decoder_parse_cte ( s...,31
3,chrome,None,1,static hb_language_t * language_reference ( hb...,34
4,debian,None,1,extern int name ( int ) __THROW __exctype ( is...,50



--- target value_counts ---
target
0    44498
1     4041
Name: count, dtype: int64

--- func length describe ---
count     48539.000000
mean       1260.452523
std        3653.282216
min           5.000000
25%         225.000000
50%         492.000000
75%        1164.000000
max      240935.000000
Name: func, dtype: float64


===== test_raw =====
shape: (24638, 5)
columns: ['project', 'commit_id', 'target', 'func', 'idx']

--- head() ---


,project,commit_id,target,func,idx
0,neomutt,6f163e07ae68654d7ac5268cbb7565f6df79ad85,1,enum ImapAuthRes imap_auth_cram_md5(struct Ima...,1794
1,flatpak,52346bf187b5a7f1c0fe9075b328b7ad6abe78f6,1,flatpak_proxy_client_init (FlatpakProxyClient ...,1797
2,libheif,2710c930918609caaf0a664e9c7bc3dce05d5b58,1,"static int32_t gcd(int a, int b)\n{\n if (a =...",1815
3,gpac,5ce0c906ed8599d218036b18b78e8126a496f137,1,static void naludmx_queue_param_set(GF_NALUDmx...,1823
4,flatpak,a10f52a7565c549612c92b8e736a6698a53db330,1,"setup_seccomp (FlatpakBwrap *bwrap,\n ...",1830



--- target value_counts ---
target
0    23740
1      898
Name: count, dtype: int64

--- func length describe ---
count     24638.000000
mean       1299.019726
std        3414.769986
min          12.000000
25%         201.000000
50%         460.000000
75%        1159.750000
max      113975.000000
Name: func, dtype: float64




<h2>2: run.pyでのselect関数とdata.clean後のDataFrameの確認</h2>
<h3>
ここでの処理により、以下の変化が起こった
<ul>
<li>関数のをすべて使わずに一部が抽出されている。これはresult = result.head(10000)によるもの</li>
<li>長すぎる関数は短くされた。そのことでmeanやstdの値に違いがある。これはlen_filter = dataset.func.str.len() < 1200によるものであり、結果的にmaxは1199となった</li>
</ul>

以下trainデータのは生のデータでの統計情報
--- func length describe ---
count    434696.000000
mean       1252.254978
std        3759.277196
min           0.000000
25%         223.000000
50%         489.000000
75%        1158.000000
max      484356.000000
Name: func, dtype: float64

以下はtrainデータに処理を加えた後の統計情報
--- func length describe ---
count    10000.000000
mean       512.396300
std        312.441412
min         12.000000
25%        251.750000
50%        448.000000
75%        748.000000
max       1199.000000
Name: func, dtype: float64
</h3>

In [28]:
train_filtered = data.apply_filter(train_raw, select)
valid_filtered = data.apply_filter(valid_raw, select)
test_filtered  = data.apply_filter(test_raw,  select)

# clean
train_clean = data.clean(train_filtered)
valid_clean = data.clean(valid_filtered)
test_clean  = data.clean(test_filtered)

# 公式 split なので commit_id / project を落とす（run.py と合わせる）
for df in (train_clean, valid_clean, test_clean):
    data.drop(df, ["commit_id", "project"])

show_basic_df_stats(train_clean, "train_clean")
show_basic_df_stats(valid_clean, "valid_clean")
show_basic_df_stats(test_clean,  "test_clean")


===== train_clean =====
shape: (1000, 3)
columns: ['target', 'func', 'idx']

--- head() ---


,target,func,idx
0,1,"extern int name ( int , locale_t ) __THROW __e...",0
2,1,WRITE_CLASS_ENCODER ( CephXServiceTicket ) str...,2
5,1,"int16_t vp9_ac_quant ( int qindex , int delta ...",5
6,1,static int ipvideo_decode_block_opcode_0xC ( I...,6
7,1,static __inline __uint64_t __uint64_identity (...,7



--- target value_counts ---
target
1    1000
Name: count, dtype: int64

--- func length describe ---
count    1000.000000
mean      574.042000
std       311.632612
min        51.000000
25%       303.750000
50%       543.500000
75%       831.500000
max      1197.000000
Name: func, dtype: float64


===== valid_clean =====
shape: (1000, 3)
columns: ['target', 'func', 'idx']

--- head() ---


,target,func,idx
0,1,static void set_fixed_partitioning ( VP9_COMP ...,23
1,1,"static void _HZOpen ( UConverter * cnv , UConv...",25
2,1,enum message_cte message_decoder_parse_cte ( s...,31
3,1,static hb_language_t * language_reference ( hb...,34
4,1,extern int name ( int ) __THROW __exctype ( is...,50



--- target value_counts ---
target
1    750
0    250
Name: count, dtype: int64

--- func length describe ---
count    1000.000000
mean      520.617000
std       306.951292
min        28.000000
25%       267.750000
50%       457.000000
75%       753.500000
max      1190.000000
Name: func, dtype: float64


===== test_clean =====
shape: (1000, 3)
columns: ['target', 'func', 'idx']

--- head() ---


,target,func,idx
1,1,flatpak_proxy_client_init (FlatpakProxyClient ...,1797
2,1,"static int32_t gcd(int a, int b)\n{\n if (a =...",1815
5,1,size_t compile_tree(struct filter_op **fop)\n{...,1835
7,1,cib_remote_connection_destroy(gpointer user_da...,1844
8,1,\nvoid gitn_box_del(GF_Box *s)\n{\n\tu32 i;\n\...,1845



--- target value_counts ---
target
0    711
1    289
Name: count, dtype: int64

--- func length describe ---
count    1000.000000
mean      450.752000
std       314.431849
min        28.000000
25%       184.000000
50%       361.000000
75%       671.000000
max      1197.000000
Name: func, dtype: float64




<h2>CPG pklの中身確認</h2>

In [29]:


cpg_files = sorted(Path(PATHS.cpg).glob("*.pkl"))
cpg_files



[PosixPath('data/cpg/test_0_cpg.pkl'),
 PosixPath('data/cpg/test_1_cpg.pkl'),
 PosixPath('data/cpg/train_0_cpg.pkl'),
 PosixPath('data/cpg/train_1_cpg.pkl'),
 PosixPath('data/cpg/valid_0_cpg.pkl'),
 PosixPath('data/cpg/valid_1_cpg.pkl')]

まず、ファイルをいくつか抽出し、それぞれのファイルの統計情報を見る。


In [30]:
from pathlib import Path
import pandas as pd

# CPG データの入っているフォルダを指定
cpg_dir = Path(PATHS.cpg)  # 例: Path("./data/cpg")

# 任意の pkl ファイル（最初の1つ）を読み込む
pkl_files = sorted(cpg_dir.glob("*.pkl"))
if not pkl_files:
    print("No .pkl files found in:", cpg_dir)
else:
    sample_file = pkl_files[0]
    print("Sample file:", sample_file.name)

    df = pd.read_pickle(sample_file)

    print("\n=== Columns in this DataFrame ===")
    print(list(df.columns))

    print("\n=== dtypes ===")
    print(df.dtypes)

    print("\n=== head() ===")
    print(df.head())

Sample file: test_0_cpg.pkl

=== Columns in this DataFrame ===
['target', 'func', 'idx', 'Index', 'cpg']

=== dtypes ===
target     int64
func      object
idx        int64
Index      int64
cpg       object
dtype: object

=== head() ===
   target                                               func   idx  Index  \
1       1  flatpak_proxy_client_init (FlatpakProxyClient ...  1797      1   
1       1  flatpak_proxy_client_init (FlatpakProxyClient ...  1797      1   
2       1  static int32_t gcd(int a, int b)\n{\n  if (a =...  1815      2   
2       1  static int32_t gcd(int a, int b)\n{\n  if (a =...  1815      2   
5       1  size_t compile_tree(struct filter_op **fop)\n{...  1835      5   

                                                 cpg  
1  {'functions': [{'function': 'flatpak_proxy_cli...  
1  {'functions': [{'function': '1.c:<global>', 'i...  
2  {'functions': [{'function': 'gcd', 'id': '1073...  
2  {'functions': [{'function': '2.c:<global>', 'i...  
5  {'functions': [{'functi

<h2>CPG pklファイルの基本的な統計情報
 カラムはtarget, func, idx, Index, cpgからなり、cpg内にはCPGデータが入っている。次の項でこのなかみは深堀する
</h2>

<pre>
Sample file: test_0_cpg.pkl

=== Columns in this DataFrame ===
['target', 'func', 'idx', 'Index', 'cpg']

=== dtypes ===
target     int64
func      object
idx        int64
Index      int64
cpg       object
dtype: object

=== head() ===
   target                                               func   idx  Index  \
1       1  flatpak_proxy_client_init (FlatpakProxyClient ...  1797      1   
1       1  flatpak_proxy_client_init (FlatpakProxyClient ...  1797      1   
2       1  static int32_t gcd(int a, int b)\n{\n  if (a =...  1815      2   
2       1  static int32_t gcd(int a, int b)\n{\n  if (a =...  1815      2   
5       1  size_t compile_tree(struct filter_op **fop)\n{...  1835      5   

                                                 cpg  
1  {'functions': [{'function': 'flatpak_proxy_cli...  
1  {'functions': [{'function': '1.c:<global>', 'i...  
2  {'functions': [{'function': 'gcd', 'id': '1073...  
2  {'functions': [{'function': '2.c:<global>', 'i...  
5  {'functions': [{'function': 'compile_tree', 'i...
</pre>

上の結果で=== head() ===以下で同じコードが重複しているっぽいので調べてみる。
以下の結果から重複しているようなコードでもcpgカラムのデータが違うことがわかる。

In [31]:
import json

def normalize_cpg(x):
    # dict の場合はキー順をソートして JSON 文字列に
    if isinstance(x, dict):
        try:
            return json.dumps(x, sort_keys=True)
        except TypeError:
            # JSON にできないオブジェクトが入っていた場合の保険
            return str(x)
    # dict 以外はとりあえず文字列化
    return str(x)

# 一時的なコピーを作って cpg_norm カラムを追加
df_tmp = df.copy()
df_tmp["cpg_norm"] = df_tmp["cpg"].map(normalize_cpg)

cols_with_norm = [c for c in df.columns if c != "cpg"] + ["cpg_norm"]

# ここで「全カラム（cpg_norm 含む）」が完全一致している行を探す
n_dup_all = df_tmp.duplicated(subset=cols_with_norm, keep=False).sum()
print("Number of fully duplicated rows (including 'cpg'):", n_dup_all)

if n_dup_all > 0:
    dup_all = df_tmp[df_tmp.duplicated(subset=cols_with_norm, keep=False)]
    dup_all = dup_all.sort_values(cols_with_norm)
    print("\n=== Example of fully duplicated rows (up to 20 rows) ===")
    # 元の cpg も見たいので一緒に表示
    print(dup_all[df.columns].head(20))
else:
    print("No fully duplicated rows (including 'cpg').")



Number of fully duplicated rows (including 'cpg'): 0
No fully duplicated rows (including 'cpg').


In [32]:
from pathlib import Path
import pandas as pd

# ここでは cpg_files が「CPG pkl の Path のリスト」だと仮定
# 例: cpg_files = sorted(PATHS.cpg.glob("*.pkl"))

split_prefixes = ["train_", "valid_", "test_"]
max_n_per_split = 5  # 各 split につき最大いくつのファイルを表示するか

def count_functions_in_cpg(cpg_obj):
    """1つの cpg オブジェクト内の functions 数を数えるヘルパー"""
    if isinstance(cpg_obj, dict):
        funcs = cpg_obj.get("functions", [])
        if isinstance(funcs, (list, tuple)):
            return len(funcs)
    return 0

for prefix in split_prefixes:
    print("#" * 100)
    print(f"=== Split: {prefix.rstrip('_')} (files starting with '{prefix}') ===")

    # prefix ごとにファイルを抽出
    split_files = sorted([p for p in cpg_files if p.name.startswith(prefix)])
    print("Found files:", [p.name for p in split_files])
    print()

    if not split_files:
        print(f"No {prefix}*.pkl found.\n")
        continue

    # 最大 max_n_per_split 個までチェック
    for i, cpg_sample in enumerate(split_files[:max_n_per_split], start=1):
        print("-" * 90)
        print(f"[{i}] File: {cpg_sample.name}")

        df_cpg = pd.read_pickle(cpg_sample)

        # --- target のカウント ---
        print("\n--- target value_counts ---")
        print(df_cpg["target"].value_counts())

        # --- 行数（≒ 関数の行数） ---
        num_rows = len(df_cpg)
        print(f"\nRows in this file (≈ number of CPG entries / functions): {num_rows}")

        # --- cpg['functions'] ベースで「関数数」を厳密にカウント ---
        funcs_per_row = df_cpg["cpg"].map(count_functions_in_cpg)
        total_funcs = int(funcs_per_row.sum())

        print(f"Total functions inside CPG['functions']: {total_funcs}")
        print("functions per row (min / mean / max):")
        print(funcs_per_row.describe()[["min", "mean", "max"]])
        print()

    print(f"=== End of split: {prefix.rstrip('_')} ===\n")






####################################################################################################
=== Split: train (files starting with 'train_') ===
Found files: ['train_0_cpg.pkl', 'train_1_cpg.pkl']

------------------------------------------------------------------------------------------
[1] File: train_0_cpg.pkl

--- target value_counts ---
target
1    948
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 948
Total functions inside CPG['functions']: 948
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[2] File: train_1_cpg.pkl

--- target value_counts ---
target
1    948
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 948
Total functions inside CPG['functions']: 948
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

==

train/validation/testの全てで、最初に脆弱性コードがあり、後半で脆弱性のないコードのみを含んでいる
ファイルそれぞれが９００近くの関数を含む

<pre>
####################################################################################################
=== Split: train (files starting with 'train_') ===
Found files: ['train_0_cpg.pkl', 'train_10_cpg.pkl', 'train_11_cpg.pkl', 'train_12_cpg.pkl', 'train_13_cpg.pkl', 'train_14_cpg.pkl', 'train_15_cpg.pkl', 'train_16_cpg.pkl', 'train_17_cpg.pkl', 'train_18_cpg.pkl', 'train_19_cpg.pkl', 'train_1_cpg.pkl', 'train_2_cpg.pkl', 'train_3_cpg.pkl', 'train_4_cpg.pkl', 'train_5_cpg.pkl', 'train_6_cpg.pkl', 'train_7_cpg.pkl', 'train_8_cpg.pkl', 'train_9_cpg.pkl']

------------------------------------------------------------------------------------------
[1] File: train_0_cpg.pkl

--- target value_counts ---
target
1    948
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 948
Total functions inside CPG['functions']: 948
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[2] File: train_10_cpg.pkl

--- target value_counts ---
target
1    650
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 650
Total functions inside CPG['functions']: 650
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[3] File: train_11_cpg.pkl

--- target value_counts ---
target
1    672
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 672
Total functions inside CPG['functions']: 672
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[4] File: train_12_cpg.pkl

--- target value_counts ---
target
1    665
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 665
Total functions inside CPG['functions']: 665
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[5] File: train_13_cpg.pkl

--- target value_counts ---
target
0    412
1    368
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 780
Total functions inside CPG['functions']: 780
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

=== End of split: train ===

####################################################################################################
=== Split: valid (files starting with 'valid_') ===
Found files: ['valid_0_cpg.pkl', 'valid_10_cpg.pkl', 'valid_11_cpg.pkl', 'valid_12_cpg.pkl', 'valid_13_cpg.pkl', 'valid_14_cpg.pkl', 'valid_15_cpg.pkl', 'valid_16_cpg.pkl', 'valid_17_cpg.pkl', 'valid_18_cpg.pkl', 'valid_19_cpg.pkl', 'valid_1_cpg.pkl', 'valid_2_cpg.pkl', 'valid_3_cpg.pkl', 'valid_4_cpg.pkl', 'valid_5_cpg.pkl', 'valid_6_cpg.pkl', 'valid_7_cpg.pkl', 'valid_8_cpg.pkl', 'valid_9_cpg.pkl']

------------------------------------------------------------------------------------------
[1] File: valid_0_cpg.pkl

--- target value_counts ---
target
1    879
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 879
Total functions inside CPG['functions']: 879
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[2] File: valid_10_cpg.pkl

--- target value_counts ---
target
0    901
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 901
Total functions inside CPG['functions']: 901
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[3] File: valid_11_cpg.pkl

--- target value_counts ---
target
0    908
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 908
Total functions inside CPG['functions']: 908
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[4] File: valid_12_cpg.pkl

--- target value_counts ---
target
0    910
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 910
Total functions inside CPG['functions']: 910
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[5] File: valid_13_cpg.pkl

--- target value_counts ---
target
0    911
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 911
Total functions inside CPG['functions']: 911
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

=== End of split: valid ===

####################################################################################################
=== Split: test (files starting with 'test_') ===
Found files: ['test_0_cpg.pkl', 'test_10_cpg.pkl', 'test_11_cpg.pkl', 'test_12_cpg.pkl', 'test_13_cpg.pkl', 'test_14_cpg.pkl', 'test_15_cpg.pkl', 'test_16_cpg.pkl', 'test_17_cpg.pkl', 'test_18_cpg.pkl', 'test_19_cpg.pkl', 'test_1_cpg.pkl', 'test_2_cpg.pkl', 'test_3_cpg.pkl', 'test_4_cpg.pkl', 'test_5_cpg.pkl', 'test_6_cpg.pkl', 'test_7_cpg.pkl', 'test_8_cpg.pkl', 'test_9_cpg.pkl']

------------------------------------------------------------------------------------------
[1] File: test_0_cpg.pkl

--- target value_counts ---
target
1    527
0    389
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 916
Total functions inside CPG['functions']: 916
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[2] File: test_10_cpg.pkl

--- target value_counts ---
target
0    927
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 927
Total functions inside CPG['functions']: 927
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[3] File: test_11_cpg.pkl

--- target value_counts ---
target
0    929
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 929
Total functions inside CPG['functions']: 929
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[4] File: test_12_cpg.pkl

--- target value_counts ---
target
0    912
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 912
Total functions inside CPG['functions']: 912
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

------------------------------------------------------------------------------------------
[5] File: test_13_cpg.pkl

--- target value_counts ---
target
0    931
Name: count, dtype: int64

Rows in this file (≈ number of CPG entries / functions): 931
Total functions inside CPG['functions']: 931
functions per row (min / mean / max):
min     1.0
mean    1.0
max     1.0
Name: cpg, dtype: float64

=== End of split: test ===
</pre>

In [33]:
from pathlib import Path
import pandas as pd

cpg_dir = Path(PATHS.cpg)   # ★ ここを Path() で包む
cpg_files = sorted(cpg_dir.glob("*.pkl"))

# 例: train_* の最初の1ファイルを読む
train_files = [p for p in cpg_files if p.name.startswith("train_")]
valid_files = [p for p in cpg_files if p.name.startswith("valid_")]
test_files  = [p for p in cpg_files if p.name.startswith("test_")]

print(train_files)
print(valid_files)
print(test_files)

df_train_cpg_sample = pd.read_pickle(train_files[0])  # 好きな index を選ぶ
df_valid_cpg_sample = pd.read_pickle(valid_files[0])
df_test_cpg_sample  = pd.read_pickle(test_files[0])

print("train sample shape:", df_train_cpg_sample.shape)
print("valid sample shape:", df_valid_cpg_sample.shape)
print("test  sample shape:", df_test_cpg_sample.shape)


[PosixPath('data/cpg/train_0_cpg.pkl'), PosixPath('data/cpg/train_1_cpg.pkl')]
[PosixPath('data/cpg/valid_0_cpg.pkl'), PosixPath('data/cpg/valid_1_cpg.pkl')]
[PosixPath('data/cpg/test_0_cpg.pkl'), PosixPath('data/cpg/test_1_cpg.pkl')]
train sample shape: (948, 5)
valid sample shape: (879, 5)
test  sample shape: (916, 5)


In [34]:

from pprint import pprint
import random
import pandas as pd

# ------------------------------
# Helper: CPGの1function(dict)を要約
# ------------------------------
def summarize_cpg_function(f: dict) -> dict:
    if not isinstance(f, dict):
        return {"type": str(type(f))}

    name = f.get("function")
    fid = f.get("id")

    has_ast = "AST" in f
    has_cfg = "CFG" in f
    has_pdg = "PDG" in f

    nodes = f.get("nodes") or f.get("node_list")
    edges = f.get("edges") or f.get("edge_list")
    n_nodes = len(nodes) if isinstance(nodes, (list, tuple)) else None
    n_edges = len(edges) if isinstance(edges, (list, tuple)) else None

    other_keys = sorted(
        k for k in f.keys()
        if k not in {
            "function", "id", "AST", "CFG", "PDG",
            "nodes", "edges", "node_list", "edge_list"
        }
    )

    return {
        "function": name,
        "id": fid,
        "has_AST": has_ast,
        "has_CFG": has_cfg,
        "has_PDG": has_pdg,
        "n_nodes": n_nodes,
        "n_edges": n_edges,
        "other_keys": other_keys,
    }


# ------------------------------
# メイン: df_cpg からランダムに数行サンプル
# ------------------------------

print("==== train ファイル一つからランダムに数行サンプル　抽出された一つ一つは一つの関数を表す ===")
n_rows = 3
sampled = df_train_cpg_sample.sample(n=min(n_rows, len(df_cpg)), random_state=None)

for row_idx, row in sampled.iterrows():
    print("=" * 100)
    print(f"=== Row index: {row_idx} ===")
    print("target:", row["target"])
    print("idx   :", row["idx"])

    # ------------------------------------------------------------------
    # func (ソースコード) の表示
    # ------------------------------------------------------------------
    print("\n--- func (source code) ---")
    func_src = row["func"]
    if isinstance(func_src, pd.Series):
        # 念のための保険（通常はここには来ないはず）
        func_src = func_src.iloc[0]
    func_str = str(func_src)
    print(func_str[:500], "..." if len(func_str) > 500 else "")

    # ------------------------------------------------------------------
    # cpg の基本情報
    # ------------------------------------------------------------------
    cpg_obj = row["cpg"]
    print("\n--- type(cpg_obj):", type(cpg_obj), "---")

    # 万一 Series や list のことがあれば、先頭要素を取る保険
    if isinstance(cpg_obj, pd.Series):
        cpg_obj = cpg_obj.iloc[0]
    elif isinstance(cpg_obj, (list, tuple)):
        cpg_obj = cpg_obj[0]

    if not isinstance(cpg_obj, dict):
        print("cpg は dict ではないので、そのまま pprint します。")
        pprint(cpg_obj)
        continue

    print("\n--- top-level keys in cpg ---")
    print(list(cpg_obj.keys()))

    funcs = cpg_obj.get("functions", [])
    if not isinstance(funcs, list):
        print("\n[cpg['functions']] が list ではないので、そのまま表示します。")
        pprint(funcs)
        continue

    # ------------------------------------------------------------------
    # functions summary
    # ------------------------------------------------------------------
    print("\n=== functions summary ===")
    num_funcs = len(funcs)
    print("num functions in this CPG:", num_funcs)

    if num_funcs == 0:
        print("functions が空です")
        continue

    # 関数名一覧（最大10件）
    func_names = [
        f.get("function") for f in funcs
        if isinstance(f, dict) and isinstance(f.get("function"), str)
    ]
    print("\n- function names (up to 10):")
    for name in func_names[:10]:
        print("   •", name)
    if len(func_names) > 10:
        print(f"   ... (+{len(func_names) - 10} more)")

    # AST/CFG/PDG の有無をざっくり集計
    has_ast_count = sum(1 for f in funcs if isinstance(f, dict) and "AST" in f)
    has_cfg_count = sum(1 for f in funcs if isinstance(f, dict) and "CFG" in f)
    has_pdg_count = sum(1 for f in funcs if isinstance(f, dict) and "PDG" in f)
    print("\n- structure availability:")
    print(f"   AST: {has_ast_count}/{num_funcs}")
    print(f"   CFG: {has_cfg_count}/{num_funcs}")
    print(f"   PDG: {has_pdg_count}/{num_funcs}")

    # nodes/edges のだいたいの規模感
    node_lens = []
    edge_lens = []
    for f in funcs:
        if not isinstance(f, dict):
            continue
        nodes = f.get("nodes") or f.get("node_list")
        edges = f.get("edges") or f.get("edge_list")
        if isinstance(nodes, (list, tuple)):
            node_lens.append(len(nodes))
        if isinstance(edges, (list, tuple)):
            edge_lens.append(len(edges))

    if node_lens or edge_lens:
        print("\n- nodes / edges size (min / mean / max):")
        if node_lens:
            print(
                f"   nodes: min={min(node_lens)}, "
                f"mean={sum(node_lens)/len(node_lens):.1f}, "
                f"max={max(node_lens)}"
            )
        if edge_lens:
            print(
                f"   edges: min={min(edge_lens)}, "
                f"mean={sum(edge_lens)/len(edge_lens):.1f}, "
                f"max={max(edge_lens)}"
            )

    # ------------------------------------------------------------------
    # ランダムにいくつか function をピックして詳細表示
    # ------------------------------------------------------------------
    
    print("func を表示")
    for i, f in enumerate(funcs):
        print(f"\n[function #{i}]")
        if not isinstance(f, dict):
            print("  (not dict) ->", type(f))
            pprint(f)
            continue

        summary = summarize_cpg_function(f)
        print("  function:", summary["function"])
        print("  id      :", summary["id"])
        print(
            "  AST/CFG/PDG:",
            f"{'Y' if summary['has_AST'] else '-'} / "
            f"{'Y' if summary['has_CFG'] else '-'} / "
            f"{'Y' if summary['has_PDG'] else '-'}",
        )
        print("  n_nodes :", summary["n_nodes"])
        print("  n_edges :", summary["n_edges"])

        if summary["other_keys"]:
            print("  other keys:", summary["other_keys"])



==== train ファイル一つからランダムに数行サンプル　抽出された一つ一つは一つの関数を表す ===
=== Row index: 205 ===
target: 1
idx   : 230

--- func (source code) ---
static inline void decompose_current_character ( const hb_ot_shape_normalize_context_t * c , bool shortest ) {
 hb_buffer_t * const buffer = c -> buffer ;
 hb_codepoint_t glyph ;
 if ( shortest && c -> font -> get_glyph ( buffer -> cur ( ) . codepoint , 0 , & glyph ) ) next_char ( buffer , glyph ) ;
 else if ( decompose ( c , shortest , buffer -> cur ( ) . codepoint ) ) skip_char ( buffer ) ;
 else if ( ! shortest && c -> font -> get_glyph ( buffer -> cur ( ) . codepoint , 0 , & glyph ) ) next_c ...

--- type(cpg_obj): <class 'dict'> ---

--- top-level keys in cpg ---
['functions']

=== functions summary ===
num functions in this CPG: 1

- function names (up to 10):
   • decompose_current_character

- structure availability:
   AST: 1/1
   CFG: 1/1
   PDG: 1/1
func を表示

[function #0]
  function: decompose_current_character
  id      : 107374182562
  AST/CFG/PDG

次にcpgカラム内をもっと詳しく見ていく

In [36]:
from pprint import pprint
import itertools

# さっき取得した funcs / f0 を再利用
funcs = cpg_obj.get("functions", [])
func_idx = 0
if len(funcs) == 0:
    raise ValueError("functions が空です")

f0 = funcs[func_idx]

print(f"=== Deep dive into functions[{func_idx}] ===")
print("keys:", list(f0.keys()))
print()

# 基本情報
print("[function]")
print("  name :", f0.get("function"))
print("  id   :", f0.get("id"))
print()

# AST / CFG / PDG を順番に見る
for gname in ["AST", "CFG", "PDG"]:
    g = f0.get(gname, None)
    print(f"=== {gname} ===")
    print("type:", type(g))

    if isinstance(g, (list, tuple)):
        print(f"len: {len(g)}")
        print(f"\n--- {gname}[0:5] ---")
        for i, item in enumerate(itertools.islice(g, 5)):
            print(f"[{gname}[{i}]] type={type(item)}")
            pprint(item)
            print("-" * 40)

            # さらに中身が dict のときはキー一覧も出す
            if isinstance(item, dict):
                print(f"  keys in {gname}[{i}]:", list(item.keys()))
        print()
    elif isinstance(g, dict):
        print(f"{gname} is a dict with keys:", list(g.keys()))
        print(f"\n--- {gname} sample ---")
        pprint(g)
        print()
    else:
        print(f"{gname} is None or unknown format:", g)
        print()


=== Deep dive into functions[0] ===
keys: ['function', 'id', 'AST', 'CFG', 'PDG']

[function]
  name : vp9_fdct16x16_1_c
  id   : 107374182949

=== AST ===
type: <class 'list'>
len: 53

--- AST[0:5] ---
[AST[0]] type=<class 'dict'>
{'edges': [],
 'id': '111669150417',
 'label': 'METHOD_PARAMETER_IN',
 'properties': {'CODE': 'const int16_t * input',
                'COLUMN_NUMBER': 26,
                'LINE_NUMBER': 1},
 'type': 'MethodParameterIn'}
----------------------------------------
  keys in AST[0]: ['id', 'label', 'type', 'properties', 'edges']
[AST[1]] type=<class 'dict'>
{'edges': [],
 'id': '111669150418',
 'label': 'METHOD_PARAMETER_IN',
 'properties': {'CODE': 'int16_t * output',
                'COLUMN_NUMBER': 50,
                'LINE_NUMBER': 1},
 'type': 'MethodParameterIn'}
----------------------------------------
  keys in AST[1]: ['id', 'label', 'type', 'properties', 'edges']
[AST[2]] type=<class 'dict'>
{'edges': [],
 'id': '111669150419',
 'label': 'METHOD_PARAME

<pre>
=== Deep dive into functions[0] ===
keys: ['function', 'id', 'AST', 'CFG', 'PDG']

[function]
  name : 0.c:<global>
  id   : 107374182400

=== AST ===
type: <class 'list'>
len: 4

--- AST[0:5] ---
[AST[0]] type=<class 'dict'>
{'edges': [{'id': '25769803776-AST-180388626432',
            'in': '180388626432',
            'out': '25769803776',
            'type': 'AST'},
           {'id': '25769803776-AST-180388626433',
            'in': '180388626433',
            'out': '25769803776',
            'type': 'AST'}],
 'id': '25769803776',
 'label': 'BLOCK',
 'properties': {'CODE': '<empty>', 'COLUMN_NUMBER': 1, 'LINE_NUMBER': 1},
 'type': 'Block'}
----------------------------------------
  keys in AST[0]: ['id', 'label', 'type', 'properties', 'edges']
[AST[1]] type=<class 'dict'>
{'edges': [{'id': '180388626432-CFG-180388626433',
            'in': '180388626433',
            'out': '180388626432',
            'type': 'CFG'}],
 'id': '180388626432',
 'label': 'UNKNOWN',
 'properties': {'CODE': ')', 'COLUMN_NUMBER': 34, 'LINE_NUMBER': 1},
 'type': 'Unknown'}
----------------------------------------
  keys in AST[1]: ['id', 'label', 'type', 'properties', 'edges']
[AST[2]] type=<class 'dict'>
{'edges': [],
 'id': '180388626433',
 'label': 'UNKNOWN',
 'properties': {'CODE': '__exctype_l ( isupper_l )',
                'COLUMN_NUMBER': 2,
                'LINE_NUMBER': 10},
 'type': 'Unknown'}
----------------------------------------
  keys in AST[2]: ['id', 'label', 'type', 'properties', 'edges']
[AST[3]] type=<class 'dict'>
{'edges': [],
 'id': '124554051584',
 'label': 'METHOD_RETURN',
 'properties': {'CODE': 'RET', 'COLUMN_NUMBER': 1, 'LINE_NUMBER': 1},
 'type': 'MethodReturn'}
----------------------------------------
  keys in AST[3]: ['id', 'label', 'type', 'properties', 'edges']

=== CFG ===
type: <class 'list'>
len: 3

--- CFG[0:5] ---
[CFG[0]] type=<class 'dict'>
{'edges': [{'id': '25769803776-AST-180388626432',
            'in': '180388626432',
            'out': '25769803776',
            'type': 'AST'},
           {'id': '25769803776-AST-180388626433',
            'in': '180388626433',
            'out': '25769803776',
            'type': 'AST'}],
 'id': '25769803776',
 'label': 'BLOCK',
 'properties': {'CODE': '<empty>', 'COLUMN_NUMBER': 1, 'LINE_NUMBER': 1},
 'type': 'Block'}
----------------------------------------
  keys in CFG[0]: ['id', 'label', 'type', 'properties', 'edges']
[CFG[1]] type=<class 'dict'>
{'edges': [{'id': '180388626432-CFG-180388626433',
            'in': '180388626433',
            'out': '180388626432',
            'type': 'CFG'}],
 'id': '180388626432',
 'label': 'UNKNOWN',
 'properties': {'CODE': ')', 'COLUMN_NUMBER': 34, 'LINE_NUMBER': 1},
 'type': 'Unknown'}
----------------------------------------
  keys in CFG[1]: ['id', 'label', 'type', 'properties', 'edges']
[CFG[2]] type=<class 'dict'>
{'edges': [],
 'id': '180388626433',
 'label': 'UNKNOWN',
 'properties': {'CODE': '__exctype_l ( isupper_l )',
                'COLUMN_NUMBER': 2,
                'LINE_NUMBER': 10},
 'type': 'Unknown'}
----------------------------------------
  keys in CFG[2]: ['id', 'label', 'type', 'properties', 'edges']

=== PDG ===
type: <class 'list'>
len: 0

--- PDG[0:5] ---
</pre>

In [13]:
from pprint import pprint
import itertools

def inspect_graph(node_list, graph_name="AST", max_nodes=10):
    """AST/CFG/PDG の node_list をざっくり可視化するユーティリティ"""
    print(f"=== {graph_name} summary ===")
    print("num_nodes   :", len(node_list))
    total_edges = sum(len(n.get("edges", [])) for n in node_list)
    print("total_edges :", total_edges)
    print()

    for i, n in enumerate(itertools.islice(node_list, max_nodes)):
        nid   = n.get("id")
        label = n.get("label")
        ntype = n.get("type")
        props = n.get("properties", {}) or {}

        print(f"[{graph_name} node #{i}] id={nid}, label={label}, type={ntype}")
        print(f"  CODE      : {props.get('CODE')!r}")
        print(f"  LOCATION  : line={props.get('LINE_NUMBER')}, col={props.get('COLUMN_NUMBER')}")
        edges = n.get("edges", [])
        print(f"  num_edges : {len(edges)}")

        # エッジの中身も少し
        for e in itertools.islice(edges, 5):
            print(f"    edge: type={e.get('type')}  out={e.get('out')} -> in={e.get('in')}")
        print("-" * 60)
    print()


In [14]:
funcs = cpg_obj.get("functions", [])
f0 = funcs[0]

print("function name:", f0.get("function"))
print("function id  :", f0.get("id"))
print()

inspect_graph(f0.get("AST", []), "AST")
inspect_graph(f0.get("CFG", []), "CFG")
inspect_graph(f0.get("PDG", []), "PDG")


function name: qemu_dummy_cpu_thread_fn
function id  : 107374182807

=== AST summary ===
num_nodes   : 95
total_edges : 176

[AST node #0] id=111669150232, label=METHOD_PARAMETER_IN, type=MethodParameterIn
  CODE      : 'void * arg'
  LOCATION  : line=1, col=42
  num_edges : 0
------------------------------------------------------------
[AST node #1] id=25769804639, label=BLOCK, type=Block
  CODE      : '{\n # ifdef _WIN32 fprintf ( stderr , "qtest is not supported under Windows\\n" ) ;\n exit ( 1 ) ;\n # else CPUState * cpu = arg ;\n sigset_t waitset ;\n int r ;\n qemu_mutex_lock_iothread ( ) ;\n qemu_thread_get_self ( cpu -> thread ) ;\n cpu -> thread_id = qemu_get_thread_id ( ) ;\n sigemptyset ( & waitset ) ;\n sigaddset ( & waitset , SIG_IPI ) ;\n cpu -> created = true ;\n qemu_cond_signal ( & qemu_cpu_cond ) ;\n cpu_single_env = cpu -> env_ptr ;\n while ( 1 ) {\n cpu_single_env = NULL ;\n qemu_mutex_unlock_iothread ( ) ;\n do {\n int sig ;\n r = sigwait ( & waitset , & sig ) ;\n }

function name: 0.c:<global>
function id  : 107374182400

=== AST summary ===
num_nodes   : 4
total_edges : 3

[AST node #0] id=25769803776, label=BLOCK, type=Block
  CODE      : '<empty>'
  LOCATION  : line=1, col=1
  num_edges : 2
    edge: type=AST  out=25769803776 -> in=180388626432
    edge: type=AST  out=25769803776 -> in=180388626433
------------------------------------------------------------
[AST node #1] id=180388626432, label=UNKNOWN, type=Unknown
  CODE      : ')'
  LOCATION  : line=1, col=34
  num_edges : 1
    edge: type=CFG  out=180388626432 -> in=180388626433
------------------------------------------------------------
[AST node #2] id=180388626433, label=UNKNOWN, type=Unknown
  CODE      : '__exctype_l ( isupper_l )'
  LOCATION  : line=10, col=2
  num_edges : 0
------------------------------------------------------------
[AST node #3] id=124554051584, label=METHOD_RETURN, type=MethodReturn
  CODE      : 'RET'
  LOCATION  : line=1, col=1
  num_edges : 0
------------------------------------------------------------

=== CFG summary ===
num_nodes   : 3
total_edges : 3

[CFG node #0] id=25769803776, label=BLOCK, type=Block
  CODE      : '<empty>'
  LOCATION  : line=1, col=1
  num_edges : 2
    edge: type=AST  out=25769803776 -> in=180388626432
    edge: type=AST  out=25769803776 -> in=180388626433
------------------------------------------------------------
[CFG node #1] id=180388626432, label=UNKNOWN, type=Unknown
  CODE      : ')'
  LOCATION  : line=1, col=34
  num_edges : 1
    edge: type=CFG  out=180388626432 -> in=180388626433
------------------------------------------------------------
[CFG node #2] id=180388626433, label=UNKNOWN, type=Unknown
  CODE      : '__exctype_l ( isupper_l )'
  LOCATION  : line=10, col=2
  num_edges : 0
------------------------------------------------------------

=== PDG summary ===
num_nodes   : 0
total_edges : 0

<h2>ここでCPG PKLのデータ構造をまとめる</h2>

いいところまで見えてきたので、一回ここで **「CPG pkl の構造まとめ」** を整理しますね。
（train/valid/test どれも基本は同じ構造、という前提でまとめます）

---

## 1. pkl ファイル 1個の中身 = `pandas.DataFrame`

`train_xxx_cpg.pkl` を `pd.read_pickle()` すると、だいたいこういう DataFrame が得られます：

**カラム構造（代表例）：**

* `target` : `0` or `1`

  * ラベル（非脆弱 / 脆弱）
* `func` : `str`

  * C/C++/PHP などの **関数ソースコード文字列**
* `idx` : `int`

  * 関数ごとの ID（元データ側のインデックス）
* `Index` : `int`

  * 元の DataFrame 由来のインデックス（意味はあまり重要ではない）
* `cpg` : `dict`

  * **この関数（or ファイル）に対応する CPG 情報が丸ごと入った辞書**

✔ DataFrame の **1 行 ≒ 1つの CPG エントリ（ほぼ 1 関数単位）** とみなせる。

---

## 2. 各行の `cpg` カラムの中身 = 辞書（dict）

`row["cpg"]` は基本的にこういう形の dict：

```python
{
    "functions": [ f0, f1, f2, ... ]  # list
}
```

* トップレベルキーは少なくとも `functions` がある
* `functions` は **関数（or グローバルスコープ）ごとの CPG** のリスト

👉 多くの行では `functions` の長さは 1
　（= その行は “1 関数 CPG” を持っている）

---

## 3. `functions[*]` の中身 = 「1関数の CPG」

`f = cpg["functions"][i]` は 1つの「関数（またはグローバルスコープ）」を表す dict で、
典型的にはこんなキーを持っています：

```python
{
    "function": "ilbc_decode_frame"  # or "408.c:<global>" など
    "id": "107374182865",            # Joern 内部のノードID

    "AST": [...],   # 抽象構文木のノードリスト
    "CFG": [...],   # 制御フローグラフのノードリスト
    "PDG": [...],   # 依存関係グラフ（空のことも多い）

    # （あれば）
    "nodes": [...], "edges": [...],
    "node_list": [...], "edge_list": [...],

    # その他の補助情報 (source, file, etc.)
    ...
}
```

### 3-1. `function` フィールドのパターン

ここが今回かなり重要でした：

* **普通の関数名**

  * 例：`"ilbc_decode_frame"`, `"handle_conn_write"`, `"gcd"`, `compile_tree`
  * → **ちゃんと関数単位で CPG が取れているケース**
* **グローバルスコープ**

  * 例：`"408.c:<global>"`, `"8949.c:<global>"`, `"0.c:<global>"`
  * → **ファイル全体（グローバル）の CPG を 1 つの “関数のようなもの” として扱っている**
* **重複などを表す特殊名**

  * 例：`"SPL_METHOD<duplicate>1"`
  * → マクロ展開や内部実装の都合で、Joern がつけた「重複回避用の疑似名」

---

## 4. `AST` / `CFG` / `PDG` の構造

それぞれ `list`（長さ N）になっていて、
**各要素は「ノード」を表す dict** です。

### 4-1. ノード dict の例

```python
{
    "id": "25769803776",     # ノードID (文字列)
    "label": "BLOCK",        # ノード種別 (Block, Unknown, MethodReturn など)
    "type": "Block",         # もう一段階詳しい型名
    "properties": {
        "CODE": "<empty>",
        "COLUMN_NUMBER": 1,
        "LINE_NUMBER": 1,
        ...                # そのノードに対応するソースコード情報
    },
    "edges": [
        {"id": "...-AST-...", "out": "25769803776", "in": "180388626432", "type": "AST"},
        ...
    ]
}
```

* AST:

  * 構文構造（Block, If, For, Return など）
* CFG:

  * 制御フロー（どのステートメントからどこに実行が飛ぶか）
  * ただし AST とノードを共用していることが多い（AST エッジが CFG ノード側に出てくるなど）
* PDG:

  * データ/制御依存グラフ
  * **空リスト（`len(PDG) = 0`）のケースも多い**

---

## 5. nodes / edges フィールドについて

`functions[i]` の中には：

* `nodes` / `edges`
* または `node_list` / `edge_list`

といったフィールドが **ない / None** になっている場合が多い。

👉 あなたのパイプラインでは：

* AST / CFG / PDG の `id` と `edges` 情報から
  **自前でノードリスト・エッジリスト（PyG 用の `edge_index` 等）を組み立てる設計**になっている
  （`nodes_to_input` や Joern → PyG 変換コードで対応）

---

## 6. グローバル関数 CPG と非グローバル関数 CPG

実際に集計した結果：

* この pkl の中には

  * **非グローバル関数 CPG**（`function` が普通の関数名のもの）が **431 行**
  * その他（`"XXXX.c:<global>"` など、ファイル単位 CPG）も多数
* ランダムサンプルでは：

  * `ilbc_decode_frame` → 正常な関数 CPG
  * `408.c:<global>` → グローバルスコープ CPG
  * `SPL_METHOD<duplicate>1` → マクロ展開由来の特殊関数名

👉 つまり、**この pkl には「関数単位」と「ファイル全体(global)」が混在**している。

学習用に **関数レベル CPG だけを使いたい場合**は：

* `function` 名が `"*.c:<global>"` で終わるものを除外する
  といった前処理フィルタが有効。

---

## 7. 行レベルで見ると何が揃っているか

1 行（= 1 CPG エントリ）には、だいたい以下が揃っている：

* `func` : 関数のソースコード（テキスト）
* `cpg["functions"][0]` : その関数（or ファイル）の CPG

  * `function` : 関数名 or `<global>`
  * `AST` / `CFG` / `PDG` : グラフ構造のベース
* `target` : 脆弱性ラベル

つまり：

> **「ソースコード + その CPG + ラベル」**
> が 1 行にまとまっている、学習には理想的なフォーマットになっている。

---

## 8. ざっくり一言まとめ

* pkl 1個 → `DataFrame`（行ごとに 1 関数 or グローバル CPG）
* 各行には：

  * `func`（生のソースコード）
  * `cpg`（`functions` リストを持つ dict）
  * `target`（ラベル）
* `cpg["functions"][i]` には：

  * `function` 名
  * `AST` / `CFG` / `PDG`（グラフのノード dict のリスト）
  * （場合により）`nodes` / `edges` など
* `function` 名は：

  * 通常の関数名 or `"XXXX.c:<global>"` or 特殊名（`<duplicate>`）
* PDG は空のこともある
* ノード/エッジリストは自前で構築する前提

---

この構造を前提に、

* 「関数レベル CPG だけを抽出して新しい pkl を作る」
* 「AST/CFG/PDG から PyG の `Data` を作る」
* 「CodeBERT のシーケンス情報と、R-GCN のグラフ情報を結合する」

という実装に素直につなげられます 👍

次にやりたかったら：

* **関数レベルだけフィルタした train/valid/test の pkl を一括で作るスクリプト**
* **AST/CFG/PDG → edge_index + edge_type 生成のテンプレ**

も一気に書きます。


<h2>Token pklの中身確認</h2>

In [15]:
# セル7: PATHS.tokens 内の pkl を確認
token_files = sorted(Path(PATHS.tokens).glob("*.pkl"))
token_files


[PosixPath('data/tokens/train_1_cpg_tokens.pkl')]

In [16]:
# セル8: token pkl の一つを覗く
if token_files:
    token_sample = token_files[0]
    print("Sample tokens pkl:", token_sample)
    df_tokens = pd.read_pickle(token_sample)
    show_basic_df_stats(df_tokens, f"Tokens DataFrame: {token_sample.name}")
else:
    print("No token pkl files found in", PATHS.tokens)


Sample tokens pkl: data/tokens/train_1_cpg_tokens.pkl
===== Tokens DataFrame: train_1_cpg_tokens.pkl =====
shape: (948, 2)
columns: ['tokens', 'func']

--- head() ---


,tokens,func
1066,"[static, int, FUN1, (, VAR1, *, VAR2, ,, int, ...",static int yop_paint_block ( YopDecContext * s...
1066,"[static, int, FUN1, (, VAR1, *, VAR2, ,, int, ...",static int yop_paint_block ( YopDecContext * s...
1069,"[int, FUN1, (, VAR1, *, VAR2, ,, int, VAR3, ,,...","int parse_CCategSpec ( tvbuff_t * tvb , int of..."
1069,"[int, FUN1, (, VAR1, *, VAR2, ,, int, VAR3, ,,...","int parse_CCategSpec ( tvbuff_t * tvb , int of..."
1070,"[static, void, FUN1, (, struct, VAR1, *, VAR1,...",static void test_show_object ( struct object *...



--- func length describe ---
count     948.000000
mean      591.632911
std       317.744693
min        51.000000
25%       316.000000
50%       557.000000
75%       864.000000
max      1197.000000
Name: func, dtype: float64




<h2>input pkl (PyG Data)の確認</h2>

In [17]:
# セル9: PATHS.input 内の *_input.pkl を一覧
input_files = sorted(Path(PATHS.input).glob("*.pkl"))
print(input_files)


[]


In [18]:
# セル10: 例として train_*_input.pkl を確認
train_input_files = [p for p in input_files if p.name.startswith("train_")]
if train_input_files:
    input_sample0 = train_input_files[0]
    input_sample4 = train_input_files[4]
    input_sample6 = train_input_files[6]
    print("Sample input pkl:", input_sample0)
    print("Sample input pkl:", input_sample4)
    print("Sample input pkl:", input_sample6)
    
    df_input0 = pd.read_pickle(input_sample0)
    df_input4 = pd.read_pickle(input_sample4)
    df_input6 = pd.read_pickle(input_sample6)
    show_basic_df_stats(df_input0, f"Input DataFrame: {input_sample0.name}")
    show_basic_df_stats(df_input4, f"Input DataFrame: {input_sample4.name}")
    show_basic_df_stats(df_input6, f"Input DataFrame: {input_sample6.name}")
else:
    print("No train_*_input.pkl found in", PATHS.input)


No train_*_input.pkl found in data/input/


<h2>StreamGraphDataset/BalancedStreamDatasetの中身の確認</h2>

In [19]:
# セル12: _split_file_paths で train split の input pkl を取得
train_input_paths = _split_file_paths("train")
len(train_input_paths), train_input_paths[:20]


(0, [])

In [20]:
# セル13: StreamGraphDataset から数件取り出して確認
from torch.utils.data import DataLoader
from torch_geometric.data import Batch

stream_ds = StreamGraphDataset(train_input_paths, buffer_size=32, shuffle=True)
stream_loader = DataLoader(
    stream_ds,
    batch_size=4,
    collate_fn=pyg_collate,
    num_workers=0
)

batch = next(iter(stream_loader))
print(type(batch))
print(batch)
print("num_graphs:", batch.num_graphs)
print("y:", batch.y)


StopIteration: 

In [ ]:
# セル14: BalancedStreamDataset の挙動確認（ラベル0/1が半々になっているかを見る）
from run import BalancedStreamDataset  # 既にimport済みなら不要

balanced_ds = BalancedStreamDataset(
    train_input_paths,
    batch_size=8,          # 偶数推奨
    buffer_size=128,
    shuffle=True,
)

balanced_loader = DataLoader(
    balanced_ds,
    batch_size=8,
    collate_fn=pyg_collate,
    num_workers=0
)

batch_bal = next(iter(balanced_loader))
print("Balanced batch y:", batch_bal.y.tolist())
print("label 0 count:", (batch_bal.y == 0).sum().item())
print("label 1 count:", (batch_bal.y == 1).sum().item())


Balanced batch y: [1, 1, 1, 0, 0, 1, 0, 0]
label 0 count: 4
label 1 count: 4


In [ ]:
# セル14-1: 複数バッチで 0/1 分布を確認する
import torch

num_batches_to_check = 50   # 好きな回数に変えてOK（多すぎると時間はかかる）

total_0 = 0
total_1 = 0
batch_stats = []

for b_idx, batch in enumerate(balanced_loader):
    y = batch.y
    c0 = (y == 0).sum().item()
    c1 = (y == 1).sum().item()
    total_0 += c0
    total_1 += c1
    batch_stats.append((c0, c1))
    print(f"Batch {b_idx:02d}: y={y.tolist()}  (0: {c0}, 1: {c1})")
    if b_idx + 1 >= num_batches_to_check:
        break

print("\n=== Summary over", len(batch_stats), "batches ===")
print("total label 0:", total_0)
print("total label 1:", total_1)
if (total_0 + total_1) > 0:
    print("ratio 0:", total_0 / (total_0 + total_1))
    print("ratio 1:", total_1 / (total_0 + total_1))


Batch 00: y=[0, 1, 1, 0, 1, 0, 1, 0]  (0: 4, 1: 4)
Batch 01: y=[0, 1, 0, 1, 0, 1, 0, 1]  (0: 4, 1: 4)
Batch 02: y=[1, 0, 1, 1, 0, 1, 0, 0]  (0: 4, 1: 4)
Batch 03: y=[1, 1, 0, 0, 0, 1, 1, 0]  (0: 4, 1: 4)
Batch 04: y=[0, 1, 1, 1, 1, 0, 0, 0]  (0: 4, 1: 4)
Batch 05: y=[0, 1, 1, 0, 1, 1, 0, 0]  (0: 4, 1: 4)
Batch 06: y=[0, 0, 1, 0, 1, 1, 1, 0]  (0: 4, 1: 4)
Batch 07: y=[1, 0, 1, 0, 0, 0, 1, 1]  (0: 4, 1: 4)
Batch 08: y=[0, 0, 1, 1, 0, 1, 0, 1]  (0: 4, 1: 4)
Batch 09: y=[1, 0, 0, 1, 0, 1, 0, 1]  (0: 4, 1: 4)
Batch 10: y=[1, 0, 1, 1, 0, 0, 1, 0]  (0: 4, 1: 4)
Batch 11: y=[1, 0, 0, 1, 0, 1, 0, 1]  (0: 4, 1: 4)
Batch 12: y=[0, 0, 0, 1, 0, 1, 1, 1]  (0: 4, 1: 4)
Batch 13: y=[1, 0, 0, 1, 0, 1, 1, 0]  (0: 4, 1: 4)
Batch 14: y=[1, 0, 0, 1, 0, 1, 0, 1]  (0: 4, 1: 4)
Batch 15: y=[1, 0, 0, 0, 1, 0, 1, 1]  (0: 4, 1: 4)
Batch 16: y=[1, 0, 0, 1, 1, 0, 1, 0]  (0: 4, 1: 4)
Batch 17: y=[0, 0, 1, 1, 1, 0, 1, 0]  (0: 4, 1: 4)
Batch 18: y=[1, 0, 0, 0, 0, 1, 1, 1]  (0: 4, 1: 4)
Batch 19: y=[0, 1, 0, 1, 1, 1, 

In [ ]:
# セル14-2: 1バッチ分のグラフ統計を詳しく見る
from collections import defaultdict

# 新しく1バッチ取得（上で batch_bal をすでに取っているならそれを使ってもOK）
batch_bal = next(iter(balanced_loader))

print("=== Batch info ===")
print(batch_bal)
print("num_graphs:", batch_bal.num_graphs)
print("y:", batch_bal.y.tolist())

# PyG Batch から各グラフのノード数を計算
# batch_bal.batch が [0,0,0,1,1,2,2,2,...] みたいになっている想定
node_counts = torch.bincount(batch_bal.batch).tolist()

# エッジは edge_index[0] のグラフごとに数える
edge_counts = defaultdict(int)
if hasattr(batch_bal, "edge_index") and batch_bal.edge_index is not None:
    src = batch_bal.edge_index[0]
    # 各ノードが属するグラフIDを使って、そのグラフのエッジ数をカウント
    for i in range(batch_bal.edge_index.size(1)):
        u = batch_bal.edge_index[0, i].item()
        g_id = batch_bal.batch[u].item()
        edge_counts[g_id] += 1

print("\n=== Graph-wise stats ===")
for g_id in range(batch_bal.num_graphs):
    y_g = batch_bal.y[g_id].item() if batch_bal.y.dim() == 1 else batch_bal.y[g_id]
    n_nodes = node_counts[g_id] if g_id < len(node_counts) else "?"
    n_edges = edge_counts.get(g_id, 0)
    print(f"- graph {g_id}: label={y_g}, num_nodes={n_nodes}, num_edges={n_edges}")


=== Batch info ===
DataBatch(x=[4000, 769], edge_index=[2, 948], y=[8], edge_type=[948], num_nodes=328, idx=[8], batch=[328], ptr=[9])
num_graphs: 8
y: [0, 1, 1, 1, 0, 0, 1, 0]

=== Graph-wise stats ===
- graph 0: label=0, num_nodes=24, num_edges=42
- graph 1: label=1, num_nodes=35, num_edges=137
- graph 2: label=1, num_nodes=2, num_edges=2
- graph 3: label=1, num_nodes=2, num_edges=2
- graph 4: label=0, num_nodes=115, num_edges=217
- graph 5: label=0, num_nodes=43, num_edges=180
- graph 6: label=1, num_nodes=2, num_edges=2
- graph 7: label=0, num_nodes=105, num_edges=366


In [ ]:
# セル14-3: ラベルごとにノード数・エッジ数の平均を見てみる（複数バッチ）
import math

num_batches_to_check = 50

nodes_by_label = defaultdict(list)
edges_by_label = defaultdict(list)

for b_idx, batch in enumerate(balanced_loader):
    # バッチ内の graph-wise ノード数
    node_counts = torch.bincount(batch.batch).tolist()
    edge_counts = defaultdict(int)
    if hasattr(batch, "edge_index") and batch.edge_index is not None:
        for i in range(batch.edge_index.size(1)):
            u = batch.edge_index[0, i].item()
            g_id = batch.batch[u].item()
            edge_counts[g_id] += 1

    for g_id in range(batch.num_graphs):
        y_g = int(batch.y[g_id].item()) if batch.y.dim() == 1 else int(batch.y[g_id])
        n_nodes = node_counts[g_id] if g_id < len(node_counts) else math.nan
        n_edges = edge_counts.get(g_id, 0)
        nodes_by_label[y_g].append(n_nodes)
        edges_by_label[y_g].append(n_edges)

    if b_idx + 1 >= num_batches_to_check:
        break

print("=== Node count stats per label ===")
for label, vals in nodes_by_label.items():
    if len(vals) == 0:
        continue
    print(f"label {label}: n={len(vals)}, mean={sum(vals)/len(vals):.2f}, min={min(vals)}, max={max(vals)}")

print("\n=== Edge count stats per label ===")
for label, vals in edges_by_label.items():
    if len(vals) == 0:
        continue
    print(f"label {label}: n={len(vals)}, mean={sum(vals)/len(vals):.2f}, min={min(vals)}, max={max(vals)}")


=== Node count stats per label ===
label 0: n=200, mean=54.12, min=2, max=236
label 1: n=200, mean=44.24, min=2, max=255

=== Edge count stats per label ===
label 0: n=200, mean=154.67, min=2, max=887
label 1: n=200, mean=124.26, min=2, max=950


In [ ]:
# セル14-4: describe_pyg_graph で各グラフを詳しく見る（改良版）
from utils.validate.analyze_utils import describe_pyg_graph
from torch_geometric.data import Batch
import torch

batch_bal = next(iter(balanced_loader))

print("=== Batch y ===", batch_bal.y.tolist())

# ※ DataList を先に作る
g_list = batch_bal.to_data_list()

for g_id, g in enumerate(g_list):
    print("\n==============================")
    print(f"Graph {g_id} (label={int(g.y.item())})")
    print("==============================")

    print(g)  # PyG の Data オブジェクトの基本情報（ノード数・エッジ数など）

    # ---------------------------
    # ★ 追加：data.x の中身を完全に表示
    # ---------------------------
    if hasattr(g, "x") and g.x is not None:
        print("\n--- g.x (node features) ---")
        print("type:", type(g.x))
        print("shape:", tuple(g.x.shape))

        # g.x の先頭数行だけ表示（全部見ると巨大なので）
        preview_rows = min(5, g.x.size(0))
        print("\n[g.x の先頭 {} 行]".format(preview_rows))
        print(g.x[:preview_rows])     # ★ これで CodeBERT or 埋め込みの実体が見える

        # 数値情報を追加
        print("\n[値の統計]")
        try:
            print("min:", float(g.x.min()))
            print("max:", float(g.x.max()))
            print("mean:", float(g.x.mean()))
            print("std:", float(g.x.std()))
        except Exception as e:
            print("統計表示不可:", e)
    else:
        print("\n[g.x が存在しません]")

    # ---------------------------
    # 従来の describe_pyg_graph()
    # ---------------------------
    print("\n--- describe_pyg_graph(g) ---")
    describe_pyg_graph(g)



=== Batch y === [1, 1, 0, 1, 0, 0, 1, 0]

Graph 0 (label=1)
Data(x=[500, 769], edge_index=[2, 2], y=[1], edge_type=[2], idx=[1], num_nodes=2)

--- g.x (node features) ---
type: <class 'torch.Tensor'>
shape: (500, 769)

[g.x の先頭 5 行]
tensor([[ 0.0000, -0.1421,  0.3667,  ..., -0.1918, -0.2447,  0.3195],
        [ 0.0000, -0.1421,  0.3667,  ..., -0.1918, -0.2447,  0.3195],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000]])

[値の統計]
min: -1.7225960493087769
max: 14.868678092956543
mean: 0.00016690434131305665
std: 0.0413585864007473

--- describe_pyg_graph(g) ---
y: (torch.int64, (1,))
x: (torch.float32, (500, 769))
edge_index: (torch.int64, (2, 2))
num_edges: 2
num_nodes: 2
edge_type counts: {0: 1, 1: 1}

Graph 1 (label=1)
Data(x=[500, 769], edge_index=[2, 2], y=[1], edge_type=[2], idx=[1], num_nodes=2)

--- g.x (node features) ---


In [21]:
# ============================================
# nodes.items() と node.get_code() の中身を見るデバッグセル
# ============================================
from pathlib import Path
from pprint import pprint

import pandas as pd

from utils.functions.cpg import parse_to_functions  # run.py と同じ
# nodes_to_input を使いたい場合は ↓ も import できる
# from utils.process import embeddings, process

# -----------------------------
# 1. df_cpg を用意する
# -----------------------------
if "df_cpg" not in globals():
    # まだ df_cpg を読み込んでいない場合は、サンプルの train_*_cpg.pkl を読む
    cpg_dir = Path("./data/cpg")  # 必要ならパスを調整
    cpg_files = sorted(cpg_dir.glob("train_*_cpg.pkl"))
    if not cpg_files:
        raise FileNotFoundError(f"{cpg_dir} 内に train_*_cpg.pkl が見つかりません")
    sample_file = cpg_files[0]
    print(f"[INFO] Loading CPG DataFrame from: {sample_file}")
    df_cpg = pd.read_pickle(sample_file)

print("\n=== df_cpg columns ===")
print(df_cpg.columns.tolist())

# -----------------------------
# 2. 調べたい行・関数を選ぶ
# -----------------------------
row_idx = df_cpg.index[0]   # ★ 調べたい行があればここを変えてOK
row = df_cpg.loc[row_idx]

print(f"\n=== row index = {row_idx} の基本情報 ===")
for col in df_cpg.columns:
    print(f"- {col}: type={type(row[col])}")

# run.py と同じく row.cpg から関数ごとの nodes を生成
max_nodes = 500  # context.nodes_dim と同じ値に合わせる
func_dicts = parse_to_functions(row.cpg, max_nodes=max_nodes)
print(f"\n[INFO] parse_to_functions から得た関数数: {len(func_dicts)}")

if not func_dicts:
    raise ValueError("func_dicts が空です。row.cpg に関数が含まれていない可能性があります。")

func_idx = 0  # ★ 何番目の関数を調べるか。必要なら 1,2,... に変更
nodes = func_dicts[func_idx]  # OrderedDict[node_id -> node]

print(f"\n=== 関数 {func_idx} の nodes 概要 ===")
print(f"type(nodes): {type(nodes)}")
print(f"len(nodes):  {len(nodes)}")

# -----------------------------
# 3. nodes.items() の中身を確認
# -----------------------------
print("\n=== nodes.items() の中身（先頭20件） ===\n")

for i, (node_id, node) in enumerate(nodes.items()):
    if i >= 20:  # ★ 20個以上見たければこの値を増やす
        print("... (以降省略) ...")
        break

    print(f"--- Node {i} ---")
    print(f"node_id: {node_id}")

    # get_code / line / column は Node or _NodeStub の実装に依存
    # 例: node.get_code(), node.code など
    code = None
    try:
        if hasattr(node, "get_code"):
            code = node.get_code()
        elif hasattr(node, "code"):
            code = node.code
    except Exception as e:
        code = f"ERROR in get_code(): {e!r}"

    line = None
    col = None
    try:
        if hasattr(node, "get_line_number"):
            line = node.get_line_number()
        elif hasattr(node, "line_number"):
            line = node.line_number
    except Exception as e:
        line = f"ERROR in line_number: {e!r}"

    try:
        if hasattr(node, "get_column_number"):
            col = node.get_column_number()
        elif hasattr(node, "column_number"):
            col = node.column_number
    except Exception as e:
        col = f"ERROR in column_number: {e!r}"

    # type / label / order / edges も確認
    n_type  = getattr(node, "type", None)
    label   = getattr(node, "label", None)
    order   = getattr(node, "order", None)
    edges   = getattr(node, "edges", {})

    print("code :", repr(code))
    print("type :", n_type)
    print("label:", label)
    print("line :", line)
    print("col  :", col)
    print("order:", order)
    print("edges count:", len(edges) if isinstance(edges, dict) else "N/A")
    print("---------------------------\n")



=== df_cpg columns ===
['target', 'func', 'idx', 'Index', 'cpg']

=== row index = 987 の基本情報 ===
- target: type=<class 'pandas.core.series.Series'>
- func: type=<class 'pandas.core.series.Series'>
- idx: type=<class 'pandas.core.series.Series'>
- Index: type=<class 'pandas.core.series.Series'>
- cpg: type=<class 'pandas.core.series.Series'>

[INFO] parse_to_functions から得た関数数: 0


ValueError: func_dicts が空です。row.cpg に関数が含まれていない可能性があります。

In [22]:
from pathlib import Path
from pprint import pprint
import pandas as pd

from utils.functions.cpg import parse_to_functions

# df_cpg がまだなければ読み込み
if "df_cpg" not in globals():
    cpg_dir = Path("./data/cpg")
    cpg_files = sorted(cpg_dir.glob("train_*_cpg.pkl"))
    if not cpg_files:
        raise FileNotFoundError(f"{cpg_dir} 内に train_*_cpg.pkl が見つかりません")
    sample_file = cpg_files[0]
    print(f"[INFO] Loading CPG DataFrame from: {sample_file}")
    df_cpg = pd.read_pickle(sample_file)

print("=== df_cpg columns ===", df_cpg.columns.tolist(), "\n")

max_nodes = 500

found = False
max_rows_to_scan = 400   # 必要なら増やしてOK

for row_idx, row in df_cpg.iloc[:max_rows_to_scan].iterrows():
    func_dicts = parse_to_functions(row.cpg, max_nodes=max_nodes)
    if not func_dicts:
        continue

    for func_idx, nodes in enumerate(func_dicts):
        # code が空でないノードを集める
        non_empty = []
        for node_id, node in nodes.items():
            code = None
            try:
                if hasattr(node, "get_code"):
                    code = node.get_code()
                elif hasattr(node, "code"):
                    code = node.code
            except Exception:
                code = None

            if isinstance(code, str) and code.strip():
                non_empty.append((node_id, node, code))

        if non_empty:
            print(f"=== 見つかった row_idx={row_idx}, func_idx={func_idx} ===")
            print("func (元コード) の一部:")
            print(row.func.splitlines()[0][:120], "...")
            print(f"nodes 全体数: {len(nodes)}, code が空でないノード数: {len(non_empty)}\n")

            # 先頭 10 ノードだけ詳しく表示
            for i, (node_id, node, code) in enumerate(non_empty[:10]):
                print(f"--- Node {i} (node_id={node_id}) ---")
                print("code:", repr(code))

                n_type = getattr(node, "type", None)
                label  = getattr(node, "label", None)
                line   = getattr(node, "line_number", None)
                col    = getattr(node, "column_number", None)
                order  = getattr(node, "order", None)

                print("type :", n_type)
                print("label:", label)
                print("line :", line)
                print("col  :", col)
                print("order:", order)
                print("---------------------------\n")

            found = True
            break  # 1 個見つけたら終了

    if found:
        break

if not found:
    print(f"[INFO] 上位 {max_rows_to_scan} 行では、code が空でないノードが見つかりませんでした。")


=== df_cpg columns === ['target', 'func', 'idx', 'Index', 'cpg'] 

=== 見つかった row_idx=987, func_idx=0 ===
func (元コード) の一部:
void cJSON_Delete(cJSON *c) ...
nodes 全体数: 68, code が空でないノード数: 68

--- Node 0 (node_id=124554052484) ---
code: 'RET'
type : 1
label: MethodReturn
line : 1
col  : 1
order: 0
---------------------------

--- Node 1 (node_id=111669150449) ---
code: 'cJSON *c'
type : 1
label: MethodParameterIn
line : 1
col  : 19
order: 1
---------------------------

--- Node 2 (node_id=115964117745) ---
code: 'cJSON *c'
type : 1
label: MethodParameterOut
line : 1
col  : 19
order: 2
---------------------------

--- Node 3 (node_id=25769805612) ---
code: '{\n\tcJSON *next;\n\twhile (c)\n\t{\n\t\tnext=c->next;\n\t\tif (!(c->type&cJSON_IsReference) && c->child) cJSON_Delete(c->child);\n\t\tif (!(c->type&cJSON_IsReference) && c->valuestring) cJSON_free(c->valuestring);\n\t\tif (!(c->type&cJSON_StringIsConst) && c->string) cJSON_free(c->string);\n\t\tcJSON_free(c);\n\t\tc=next;\n\t}\n}'
type 

In [ ]:
from pprint import pprint
import itertools

row_idx = df_cpg.index[0]   # 他の行を見たければここを変えてOK
row = df_cpg.loc[row_idx]

cpg_obj = row.cpg
print("=== cpg_obj keys ===")
print(list(cpg_obj.keys()))
print()

# 1. AST ブロックの中身をざっくり見る（Joern 由来ならだいたい AST に CODE があることが多い）
ast = cpg_obj.get("AST") or cpg_obj.get("ast")  # 大文字小文字両方試す
if ast is None:
    print("AST / ast キーが見つかりませんでした")
else:
    print(f"AST length: {len(ast)}")
    if len(ast) > 0:
        print("\n--- AST[0] (最初のノード) ---")
        pprint(ast[0])
        if "properties" in ast[0]:
            print("\nAST[0]['properties'] keys:", list(ast[0]["properties"].keys()))

# 2. 再帰的に 'CODE' / 'code' というキーを探すヘルパー
def find_code_keys(obj, path="root", max_hits=20, hits=None):
    if hits is None:
        hits = []
    if len(hits) >= max_hits:
        return hits

    if isinstance(obj, dict):
        for k, v in obj.items():
            if k.lower() == "code":  # 'CODE', 'code', 'Code' など全部拾う
                hits.append((f"{path}.{k}", v))
                if len(hits) >= max_hits:
                    return hits
            find_code_keys(v, f"{path}.{k}", max_hits, hits)
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            find_code_keys(v, f"{path}[{i}]", max_hits, hits)
    return hits

print("\n=== Searching 'code' / 'CODE' keys recursively in row.cpg ===")
hits = find_code_keys(cpg_obj, max_hits=20)
if not hits:
    print("❌ 'code' / 'CODE' キーが見つかりませんでした。")
else:
    print(f"見つかった個数: {len(hits)}（最大20件まで表示）\n")
    for p, v in hits:
        print(f"- path: {p}")
        # v が長いコードの場合は先頭 80 文字だけ表示
        if isinstance(v, str):
            print("  value:", repr(v[:80] + ("..." if len(v) > 80 else "")))
        else:
            print("  value(type):", type(v))
        print()


=== cpg_obj keys ===
['functions']

AST / ast キーが見つかりませんでした

=== Searching 'code' / 'CODE' keys recursively in row.cpg ===
見つかった個数: 7（最大20件まで表示）

- path: root.functions[0].AST[0].properties.CODE
  value: '<empty>'

- path: root.functions[0].AST[1].properties.CODE
  value: ')'

- path: root.functions[0].AST[2].properties.CODE
  value: '__exctype_l ( isupper_l )'

- path: root.functions[0].AST[3].properties.CODE
  value: 'RET'

- path: root.functions[0].CFG[0].properties.CODE
  value: '<empty>'

- path: root.functions[0].CFG[1].properties.CODE
  value: ')'

- path: root.functions[0].CFG[2].properties.CODE
  value: '__exctype_l ( isupper_l )'



In [ ]:
from pprint import pprint
from collections import OrderedDict

from utils.functions.cpg import parse_function_v4_to_nodes

# 1. 適当な行と関数を選ぶ
row_idx = df_cpg.index[0]
row = df_cpg.loc[row_idx]
func_json = row.cpg["functions"][0]

print(f"=== row_idx={row_idx} の function[0] のキー ===")
print(list(func_json.keys()))
print()

# 2. AST 側のノード一覧（id と CODE）を確認
ast_list = func_json.get("AST", [])
print(f"=== AST ノード数: {len(ast_list)} ===\n")

for ast_node in ast_list[:10]:  # 先頭10件だけ
    nid = ast_node.get("id", "N/A")
    props = ast_node.get("properties", {})
    code = props.get("CODE") or props.get("code")
    line = props.get("LINE_NUMBER") or props.get("lineNumber")
    col  = props.get("COLUMN_NUMBER") or props.get("columnNumber")
    print(f"[AST] id={nid}")
    print("  CODE:", repr(code))
    print("  LINE:", line, " COL:", col)
    print()

# 3. parse_function_v4_to_nodes で _NodeStub 群を作る
nodes = parse_function_v4_to_nodes(func_json, max_nodes=500, strict_filter=False)
print(f"\n=== parse_function_v4_to_nodes → nodes 概要 ===")
print("type(nodes):", type(nodes))
print("len(nodes): ", len(nodes))
print()

# 4. AST の id と nodes の id をマッピングして、CODE と node.code を比較
#    Joern の id は "1.2.3" みたいな形式で、NodeStub は末尾だけを使っていることが多いので、
#    両方とも末尾にそろえて突き合わせる
def short_id(full_id):
    if isinstance(full_id, str):
        return full_id.split(".")[-1]
    return str(full_id)

# AST 側: short_id -> CODE
ast_code_map = {}
for ast_node in ast_list:
    nid = ast_node.get("id")
    props = ast_node.get("properties", {})
    code = props.get("CODE") or props.get("code")
    ast_code_map[short_id(nid)] = code

print("=== AST vs NodeStub の対応（先頭20ノード） ===\n")
for i, (node_id, node) in enumerate(nodes.items()):
    if i >= 20:
        print("... (以降省略) ...")
        break

    sid = short_id(node_id)
    stub_code = getattr(node, "code", None)
    ast_code  = ast_code_map.get(sid)

    print(f"[NodeStub] node_id={node_id} (short={sid})")
    print("  node.code :", repr(stub_code))
    print("  AST.CODE  :", repr(ast_code))
    print("----------\n")


=== row_idx=0 の function[0] のキー ===
['function', 'id', 'AST', 'CFG', 'PDG']

=== AST ノード数: 4 ===

[AST] id=25769803776
  CODE: '<empty>'
  LINE: 1  COL: 1

[AST] id=180388626432
  CODE: ')'
  LINE: 1  COL: 34

[AST] id=180388626433
  CODE: '__exctype_l ( isupper_l )'
  LINE: 10  COL: 2

[AST] id=124554051584
  CODE: 'RET'
  LINE: 1  COL: 1


=== parse_function_v4_to_nodes → nodes 概要 ===
type(nodes): <class 'collections.OrderedDict'>
len(nodes):  3

=== AST vs NodeStub の対応（先頭20ノード） ===

[NodeStub] node_id=180388626432 (short=180388626432)
  node.code : ''
  AST.CODE  : ')'
----------

[NodeStub] node_id=180388626433 (short=180388626433)
  node.code : ''
  AST.CODE  : '__exctype_l ( isupper_l )'
----------

[NodeStub] node_id=25769803776 (short=25769803776)
  node.code : ''
  AST.CODE  : '<empty>'
----------

